# Setup

In [ ]:
from pyspark.sql import SparkSession
from utils.config_loader import load_config
from utils.logger import setup_logging, get_logger
from transformation.bronze_writer import run_bronze_ingestion

setup_logging()
logger = get_logger(__name__)

spark = SparkSession.builder.appName("bronze_ingestion").getOrCreate()
app_config = load_config()

# Run ingestion (Chapter 2's connectors), then write Bronze
# In Fabric, this cell would instead call %run ./run_ingestion or invoke it
# as a prior pipeline activity — shown inline here for notebook self-containment.

In [ ]:
from ingestion.run_ingestion import run as run_connectors

run_connectors()  # writes JSON to data/bronze/ (local) or Files/bronze/ (Fabric)

# Cell 3 — Read the connector JSON output and write the Bronze Delta table

In [ ]:
json_dir = app_config.get("storage", "bronze_path", default="data/bronze")
table_path = "Tables/bronze_job_postings"  # Fabric Lakehouse table path

# Local dev equivalent: table_path = "data/delta/bronze_job_postings"

In [ ]:
bronze_df = run_bronze_ingestion(spark, json_dir=json_dir, table_path=table_path)
bronze_df.show(5, truncate=50)

# Cell 4 — Sanity checks before trusting this run

In [ ]:
print("Row count:", bronze_df.count())
print("Schema:")
bronze_df.printSchema()
print("Records per source:")
bronze_df.groupBy("source").count().show()